# 原 Stage1 0bp HOLD 日现金机会成本上界

只读冻结原生 Qlib 账户全部 944 日；固定路径乐观分量修复不是交易规则。

In [1]:
import hashlib,json,os
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
root=Path.cwd();card=root/'research/stage1_original_0bp_hold_cash_bound_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['source'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
source=root/src['card'];summary=json.loads((source/'summary.json').read_text());assert sha(source/'summary.json')==src['summary_sha256'] and summary['recorder_id']==src['recorder_id'] and summary['nas_readback']['verified']
artifact=source/'qlib_recorder'/src['experiment_id']/src['recorder_id']/'artifacts'/src['daily_artifact'];assert sha(artifact)==src['daily_sha256'] and summary['nas_evidence']['sha256']==src['nas_archive_sha256']
state={'status':'started','plan_sha256':sha(card/'plan.json'),'source_Recorder_loads':0,'new_model_fits':0,'new_native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
qlib.init(provider_uri=str(root/src['provider_uri']),region='us')
with R.uri_context(str(source/'qlib_recorder')):
    rec=R.get_recorder(recorder_id=src['recorder_id'],experiment_id=src['experiment_id']);daily=rec.load_object(src['daily_artifact']).copy()
state['source_Recorder_loads']=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
daily=daily.sort_values('signal_date').reset_index(drop=True);assert len(daily)==944 and daily.signal_date.is_unique
cal=pd.DatetimeIndex(D.calendar(start_time='2023-01-03',end_time='2026-10-07',freq='day'));assert len(cal)==944 and pd.DatetimeIndex(pd.to_datetime(daily.signal_date)).equals(cal)
components=['cash_opportunity_log','SOXL_selection_log','Stage1_direct_fees_log','timing_and_benchmark_residual_log'];closure=(daily[components].sum(axis=1)-daily.paired_log_excess).abs().max();assert closure<1e-12
original=json.loads((source/'result.json').read_text())['full']['relative_log_excess'];assert abs(float(daily.paired_log_excess.sum())-original)<1e-10
hold=~daily.open_execute.astype(bool);binding=daily.cap_binding.astype(bool)
repair=(-daily.cash_opportunity_log).clip(lower=0).where(hold,0.0)
window=126;rolling_excess=daily.paired_log_excess.rolling(window).sum().iloc[window-1:].reset_index(drop=True);rolling_repair=repair.rolling(window).sum().iloc[window-1:].reset_index(drop=True);rolling_hold=hold.astype(int).rolling(window).sum().iloc[window-1:].reset_index(drop=True)
assert len(rolling_excess)==len(rolling_repair)==len(rolling_hold)==819
original_positive=int((rolling_excess>0).sum());optimistic_positive=int((rolling_excess+rolling_repair>0).sum());required=int(np.ceil(0.7*819));assert original_positive==255 and required==574
none=(rolling_excess<=0)&(rolling_hold==0);no_hold_nonwinning=int(none.sum())
cohorts=[]
for name,mask in [('HOLD',hold),('EXECUTE',~hold),('HOLD_binding',hold&binding),('HOLD_nonbinding',hold&~binding),('EXECUTE_binding',~hold&binding),('EXECUTE_nonbinding',~hold&~binding)]:
    g=daily.loc[mask];cohorts.append({'cohort':name,'sessions':len(g),**{k:float(g[k].sum()) for k in components+['paired_log_excess']}})
def periods(freq):
    out=[]
    for label,g in daily.groupby(pd.to_datetime(daily.trade_date).dt.to_period(freq),sort=True):
        out.append({'period':str(label),'sessions':len(g),'HOLD_sessions':int((~g.open_execute.astype(bool)).sum()),'cash_opportunity_log':float(g.cash_opportunity_log.sum()),'paired_log_excess':float(g.paired_log_excess.sum())})
    return out
result={'decision':'read_only_Stage1_HOLD_cash_direct_repair_bound_no_policy','plan_sha256':state['plan_sha256'],'source_recorder_id':src['recorder_id'],'sessions':944,'HOLD_sessions':int(hold.sum()),'EXECUTE_sessions':int((~hold).sum()),'cap_binding_sessions':int(binding.sum()),'component_closure_max_abs':float(closure),'cohorts':cohorts,'years':periods('Y'),'quarters':periods('Q'),'rolling126_windows':819,'original_positive_windows':original_positive,'required_positive_windows':required,'HOLD_only_optimistic_direct_repair_positive_windows':optimistic_positive,'HOLD_only_optimistic_direct_repair_shortfall':max(0,required-optimistic_positive),'nonwinning_windows_without_HOLD':no_hold_nonwinning,'optimistic_repair_total_log':float(repair.sum()),'fixed_path_not_causal_account':True,'new_market_requests':0,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'new_Qlib_Recorders':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n');state.update(status='completed');(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print(json.dumps({k:result[k] for k in ['decision','HOLD_sessions','original_positive_windows','HOLD_only_optimistic_direct_repair_positive_windows','required_positive_windows','nonwinning_windows_without_HOLD']}))


{"decision": "read_only_Stage1_HOLD_cash_direct_repair_bound_no_policy", "HOLD_sessions": 147, "original_positive_windows": 255, "HOLD_only_optimistic_direct_repair_positive_windows": 450, "required_positive_windows": 574, "nonwinning_windows_without_HOLD": 0}


# 原 Stage1 0bp：开盘 HOLD 与现金机会损失

## 中文结论

只读冻结的完整 Stage1 f696 分配器加 b472 行动、风控、恢复链自有资金 **0bp 附加滑点、仍扣 IBKR Tiered 手续费**原生 Qlib 账户。全部 944 个会话、四个年度（2023、2026 为部分端点年）、16 个季度和 819 个连续 126 日窗口均保留；没有重放账户、拟合或新行情请求。逐日现金、SOXL 相对选择、费用与时点残差对 SOXX 超额的闭合误差低于 1e−12。

原账户有 **147 日 HOLD**、**797 日 EXECUTE**。全程现金机会分量为 **−0.600323 对数点**：HOLD 日 **−0.148127**，EXECUTE 日 **−0.452195**，即约四分之三的现金拖累发生在实际执行目标的日子。EXECUTE 日中，风险上限受限与未受限的现金分量分别为 **−0.222006** 和 **−0.230190**；因此问题同时存在于原目标配置和风险投影，而不能只归咎于动作模型选择等待。

预先固定的**非因果乐观上界**只在 HOLD 日把负现金分量直接改为零，正贡献、SOXL 选择、费用、持仓和后续风险状态均保持原路径。原账户 **255/819** 个 126 日窗口胜 SOXX；这个几乎不可能实现的直接修复也仅达到 **450/819**，距离合同所需 **574/819** 仍差 **124** 窗。它说明“只修 HOLD 日的现金损失”在冻结路径上不足以通过稳定门，不证明任何完整自有状态策略不可能成功；真实换仓会改变后续费用、仓位和风控反馈。

决定：`read_only_Stage1_HOLD_cash_direct_repair_bound_no_policy`。下一候选应从 **EXECUTE 日的原分配目标和受限／未受限两层现金配置**寻找决策时信息或可证伪机制；不把事后 HOLD、年份或窗口变成切换阈值。SOXX 费用后财富、回撤、CVaR、完整年、滚动窗和不确定性硬门仍需一个新完整 0bp 账户证明，本卡没有候选账户、2/5bp 或独立前向证据。
